In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
project_path = "/content/drive/MyDrive/music-success-analytics"

raw_path = project_path + "/raw"
processed_path = project_path + "/processed"
quality_reports_path = project_path + "/quality_reports"

In [ ]:
print(os.listdir(project_path))

['data', 'quality_reports', 'raw', 'processed']


In [ ]:
file_path = raw_path + "/spotify_2015_2025_85k.xlsx"

df = pd.read_excel(file_path)

df.head()

,track_id,track_name,artist_name,album_name,release_date,genre,duration_ms,popularity,danceability,energy,key,loudness,mode,instrumentalness,tempo,stream_count,country,explicit,label
0,TRK-BEBD53DA84E1,Agent every (0),Noah Rhodes,Beautiful instead,2016-04-01,Pop,234194,55,0.15,0.74,9,-32.22,0,0.436,73.12,13000,Brazil,0,Universal Music
1,TRK-6A32496762D7,Night respond,Jennifer Cole,Table,2022-04-15,Metal,375706,45,0.44,0.46,0,-14.02,0,0.223,157.74,1000,France,1,Island Records
2,TRK-47AA7523463E,Future choice whatever,Brandon Davis,Page southern,2016-02-23,Rock,289191,55,0.62,0.80,8,-48.26,1,0.584,71.03,1000,Germany,1,XL Recordings
3,TRK-25ADA22E3B06,Bad fall pick those,Corey Jones,Spring,2015-10-12,Pop,209484,51,0.78,0.98,1,-34.47,1,0.684,149.00,1000,France,0,Warner Music
4,TRK-9245F2AD996A,Husband,Mark Diaz,Great prove,2022-07-08,Indie,127435,39,0.74,0.18,10,-17.84,0,0.304,155.85,2000,United States,0,Independent


In [ ]:
df_clean = df.copy()

In [ ]:
print("Original dataset shape:", df.shape)
print("Cleaning dataset shape:", df_clean.shape)

Original dataset shape: (85000, 19)
Cleaning dataset shape: (85000, 19)


In [ ]:
df_clean.columns = (
    df_clean.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
)

In [ ]:
df_clean.columns

Index(['track_id', 'track_name', 'artist_name', 'album_name', 'release_date',
       'genre', 'duration_ms', 'popularity', 'danceability', 'energy', 'key',
       'loudness', 'mode', 'instrumentalness', 'tempo', 'stream_count',
       'country', 'explicit', 'label'],
      dtype='object')

In [ ]:
df_clean["release_date"] = pd.to_datetime(
    df_clean["release_date"],
    errors="coerce"
)

In [ ]:
df_clean["release_date"].dtype

dtype('<M8[ns]')

In [ ]:
df_clean["release_year"] = df_clean["release_date"].dt.year

In [ ]:
df_clean[["release_date", "release_year"]].head()

,release_date,release_year
0,2016-04-01,2016
1,2022-04-15,2022
2,2016-02-23,2016
3,2015-10-12,2015
4,2022-07-08,2022


In [ ]:
df_clean["duration_min"] = (df_clean["duration_ms"] / 60000).round(2)

In [ ]:
df_clean[["duration_ms", "duration_min"]].head()

,duration_ms,duration_min
0,234194,3.90
1,375706,6.26
2,289191,4.82
3,209484,3.49
4,127435,2.12


In [ ]:
df_clean.shape

(85000, 21)

In [ ]:
df_clean.isna().sum().sort_values(ascending=False)

,0
album_name,46
track_name,21
track_id,0
artist_name,0
release_date,0
genre,0
duration_ms,0
popularity,0
danceability,0
energy,0


In [ ]:
df_clean["album_name"] = df_clean["album_name"].fillna("Unknown")

In [ ]:
df_clean["album_name"].isna().sum()

np.int64(0)

In [ ]:
df_clean = df_clean.dropna(subset=["track_name"])

In [ ]:
df_clean["track_name"].isna().sum()

np.int64(0)

In [ ]:
df_clean.shape

(84979, 21)

In [ ]:
df_clean.isna().sum().sort_values(ascending=False)

,0
track_id,0
track_name,0
artist_name,0
album_name,0
release_date,0
genre,0
duration_ms,0
popularity,0
danceability,0
energy,0


Missing values were handled based on the importance of each column. Missing `album_name` values were filled with "Unknown" because album information is not essential for the main analysis. Rows with missing `track_name` values were removed because the track title is important for identification, reporting and dashboard creation.

In [ ]:
numeric_checks = {
    "popularity_out_of_range": df_clean[
        (df_clean["popularity"] < 0) | (df_clean["popularity"] > 100)
    ].shape[0],

    "danceability_out_of_range": df_clean[
        (df_clean["danceability"] < 0) | (df_clean["danceability"] > 1)
    ].shape[0],

    "energy_out_of_range": df_clean[
        (df_clean["energy"] < 0) | (df_clean["energy"] > 1)
    ].shape[0],

    "instrumentalness_out_of_range": df_clean[
        (df_clean["instrumentalness"] < 0) | (df_clean["instrumentalness"] > 1)
    ].shape[0],

    "duration_invalid": df_clean[
        df_clean["duration_ms"] <= 0
    ].shape[0],

    "tempo_invalid": df_clean[
        df_clean["tempo"] <= 0
    ].shape[0],

    "stream_count_invalid": df_clean[
        df_clean["stream_count"] < 0
    ].shape[0]
}

numeric_checks

{'popularity_out_of_range': 0,
 'danceability_out_of_range': 0,
 'energy_out_of_range': 0,
 'instrumentalness_out_of_range': 0,
 'duration_invalid': 0,
 'tempo_invalid': 0,
 'stream_count_invalid': 0}

In [ ]:
df_clean[
    (df_clean["popularity"] < 0) | (df_clean["popularity"] > 100)
].shape[0]

0

In [ ]:
df_clean[df_clean["duration_ms"] <= 0].shape[0]

0

In [ ]:
df_clean[df_clean["stream_count"] < 0].shape[0]

0

All numerical range checks returned 0 invalid records. Popularity values are within the expected 0-100 range, audio feature values are within their expected 0-1 range, and duration, tempo and stream count values are valid. No numerical cleaning was required at this stage.

In [ ]:
df_clean["explicit"].value_counts()

,count
explicit,
0,67868
1,17111


In [ ]:
df_clean["mode"].value_counts()

,count
mode,
1,42501
0,42478


In [ ]:
binary_checks = {
    "explicit_unique_values": sorted(df_clean["explicit"].unique()),
    "mode_unique_values": sorted(df_clean["mode"].unique())
}

binary_checks

{'explicit_unique_values': [np.int64(0), np.int64(1)],
 'mode_unique_values': [np.int64(0), np.int64(1)]}

In [ ]:
{
 'explicit_unique_values': [np.int64(0), np.int64(1)],
 'mode_unique_values': [np.int64(0), np.int64(1)]
}

{'explicit_unique_values': [np.int64(0), np.int64(1)],
 'mode_unique_values': [np.int64(0), np.int64(1)]}

Binary categorical variables were checked for consistency. Both `explicit` and `mode` contain only the expected values 0 and 1, so no cleaning was required for these columns.

In [ ]:
df_clean["key"].value_counts().sort_index()

,count
key,
0,6994
1,7053
2,7196
3,6976
4,7078
5,7024
6,7058
7,7033
8,7124


In [ ]:
invalid_key_values = df_clean[
    (df_clean["key"] < 0) | (df_clean["key"] > 11)
].shape[0]

invalid_key_values

0

In [ ]:
df_clean = df_clean[
    (df_clean["popularity"] >= 0) & (df_clean["popularity"] <= 100) &
    (df_clean["danceability"] >= 0) & (df_clean["danceability"] <= 1) &
    (df_clean["energy"] >= 0) & (df_clean["energy"] <= 1) &
    (df_clean["instrumentalness"] >= 0) & (df_clean["instrumentalness"] <= 1) &
    (df_clean["duration_ms"] > 0) &
    (df_clean["tempo"] > 0) &
    (df_clean["stream_count"] >= 0) &
    (df_clean["key"] >= 0) & (df_clean["key"] <= 11)
]

In [ ]:
df_clean.shape

(84979, 21)

In [ ]:
hit_threshold = df_clean["popularity"].quantile(0.75)

hit_threshold

np.float64(57.0)

In [ ]:
df_clean["is_hit"] = np.where(
    df_clean["popularity"] >= hit_threshold,
    1,
    0
)

In [ ]:
df_clean["is_hit"].value_counts()

,count
is_hit,
0,63731
1,21248


In [ ]:
df_clean["is_hit"].value_counts(normalize=True).round(2)

,proportion
is_hit,
0,0.75
1,0.25


In [ ]:
df_clean["label_type"] = np.where(
    df_clean["label"] == "Independent",
    "Independent",
    "Label"
)

In [ ]:
df_clean["label_type"].value_counts()

,count
label_type,
Label,74216
Independent,10763


In [ ]:
df_clean.shape

(84979, 23)

In [ ]:
df_clean.head()

,track_id,track_name,artist_name,album_name,release_date,genre,duration_ms,popularity,danceability,energy,...,instrumentalness,tempo,stream_count,country,explicit,label,release_year,duration_min,is_hit,label_type
0,TRK-BEBD53DA84E1,Agent every (0),Noah Rhodes,Beautiful instead,2016-04-01,Pop,234194,55,0.15,0.74,...,0.436,73.12,13000,Brazil,0,Universal Music,2016,3.90,0,Label
1,TRK-6A32496762D7,Night respond,Jennifer Cole,Table,2022-04-15,Metal,375706,45,0.44,0.46,...,0.223,157.74,1000,France,1,Island Records,2022,6.26,0,Label
2,TRK-47AA7523463E,Future choice whatever,Brandon Davis,Page southern,2016-02-23,Rock,289191,55,0.62,0.80,...,0.584,71.03,1000,Germany,1,XL Recordings,2016,4.82,0,Label
3,TRK-25ADA22E3B06,Bad fall pick those,Corey Jones,Spring,2015-10-12,Pop,209484,51,0.78,0.98,...,0.684,149.00,1000,France,0,Warner Music,2015,3.49,0,Label
4,TRK-9245F2AD996A,Husband,Mark Diaz,Great prove,2022-07-08,Indie,127435,39,0.74,0.18,...,0.304,155.85,2000,United States,0,Independent,2022,2.12,0,Independent


Final consistency filters were applied to keep only records with valid numerical ranges. Two additional analytical columns were created: `is_hit`, which identifies tracks in the top 25% of popularity, and `label_type`, which separates Independent tracks from label-backed releases.

In [ ]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 84979 entries, 0 to 84999
Data columns (total 23 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   track_id          84979 non-null  object        
 1   track_name        84979 non-null  object        
 2   artist_name       84979 non-null  object        
 3   album_name        84979 non-null  object        
 4   release_date      84979 non-null  datetime64[ns]
 5   genre             84979 non-null  object        
 6   duration_ms       84979 non-null  int64         
 7   popularity        84979 non-null  int64         
 8   danceability      84979 non-null  float64       
 9   energy            84979 non-null  float64       
 10  key               84979 non-null  int64         
 11  loudness          84979 non-null  float64       
 12  mode              84979 non-null  int64         
 13  instrumentalness  84979 non-null  float64       
 14  tempo             84979 non

In [ ]:
df_clean.isna().sum().sort_values(ascending=False)

,0
track_id,0
track_name,0
artist_name,0
album_name,0
release_date,0
genre,0
duration_ms,0
popularity,0
danceability,0
energy,0


In [ ]:
df_clean.duplicated().sum()

np.int64(0)

In [ ]:
df_clean.columns

Index(['track_id', 'track_name', 'artist_name', 'album_name', 'release_date',
       'genre', 'duration_ms', 'popularity', 'danceability', 'energy', 'key',
       'loudness', 'mode', 'instrumentalness', 'tempo', 'stream_count',
       'country', 'explicit', 'label', 'release_year', 'duration_min',
       'is_hit', 'label_type'],
      dtype='object')

In [ ]:
final_columns = [
    "track_id",
    "track_name",
    "artist_name",
    "album_name",
    "release_date",
    "release_year",
    "genre",
    "duration_ms",
    "duration_min",
    "popularity",
    "is_hit",
    "danceability",
    "energy",
    "key",
    "loudness",
    "mode",
    "instrumentalness",
    "tempo",
    "stream_count",
    "country",
    "explicit",
    "label",
    "label_type"
]

df_clean = df_clean[final_columns]

In [ ]:
df_clean.head()

,track_id,track_name,artist_name,album_name,release_date,release_year,genre,duration_ms,duration_min,popularity,...,key,loudness,mode,instrumentalness,tempo,stream_count,country,explicit,label,label_type
0,TRK-BEBD53DA84E1,Agent every (0),Noah Rhodes,Beautiful instead,2016-04-01,2016,Pop,234194,3.90,55,...,9,-32.22,0,0.436,73.12,13000,Brazil,0,Universal Music,Label
1,TRK-6A32496762D7,Night respond,Jennifer Cole,Table,2022-04-15,2022,Metal,375706,6.26,45,...,0,-14.02,0,0.223,157.74,1000,France,1,Island Records,Label
2,TRK-47AA7523463E,Future choice whatever,Brandon Davis,Page southern,2016-02-23,2016,Rock,289191,4.82,55,...,8,-48.26,1,0.584,71.03,1000,Germany,1,XL Recordings,Label
3,TRK-25ADA22E3B06,Bad fall pick those,Corey Jones,Spring,2015-10-12,2015,Pop,209484,3.49,51,...,1,-34.47,1,0.684,149.00,1000,France,0,Warner Music,Label
4,TRK-9245F2AD996A,Husband,Mark Diaz,Great prove,2022-07-08,2022,Indie,127435,2.12,39,...,10,-17.84,0,0.304,155.85,2000,United States,0,Independent,Independent


In [ ]:
output_path = processed_path + "/spotify_tracks_clean.csv"

df_clean.to_csv(output_path, index=False)

In [ ]:
os.listdir(processed_path)

['spotify_tracks_clean.csv']

In [ ]:
cleaning_summary = {
    "original_rows": df.shape[0],
    "original_columns": df.shape[1],
    "final_rows": df_clean.shape[0],
    "final_columns": df_clean.shape[1],
    "removed_rows": df.shape[0] - df_clean.shape[0],
    "missing_values_after_cleaning": int(df_clean.isna().sum().sum()),
    "duplicated_track_ids_after_cleaning": int(df_clean["track_id"].duplicated().sum())
}

cleaning_summary

{'original_rows': 85000,
 'original_columns': 19,
 'final_rows': 84979,
 'final_columns': 23,
 'removed_rows': 21,
 'missing_values_after_cleaning': 0,
 'duplicated_track_ids_after_cleaning': 0}

In [ ]:
cleaning_summary_df = pd.DataFrame(
    list(cleaning_summary.items()),
    columns=["metric", "value"]
)

cleaning_summary_df.to_csv(
    quality_reports_path + "/spotify_cleaning_summary.csv",
    index=False
)

In [ ]:
os.listdir(quality_reports_path)

['data_dictionary.xlsx',
 'spotify_data_quality_report.csv',
 'spotify_cleaning_summary.csv']

## Cleaning Summary

The cleaning process started from the raw Spotify dataset containing 85,000 rows and 19 columns.  
Missing `album_name` values were replaced with "Unknown", while rows with missing `track_name` values were removed.  
Additional analytical columns were created: `release_year`, `duration_min`, `is_hit`, and `label_type`.  
Final consistency checks confirmed that the cleaned dataset contains no missing values, no duplicated `track_id` values, and valid numerical ranges.

The cleaned dataset was saved as `spotify_tracks_clean.csv` in the `processed` folder.